# XAI-guided weighted retraining orchestrator

This notebook runs the `A0/A1/A2/A4/A5` CLI experiment with the active Jupyter kernel and presents its artifacts. Start with a CUDA smoke run, then change `SMOKE` to `False` for the full validation experiment. The expensive command is guarded by `RUN_EXPERIMENT`.

## 1. Experiment controls

`auto` progress produces a live bar in a terminal and readable periodic updates in this captured notebook process. Test evaluation is intentionally blocked by the runner until the selected configuration has `frozen=true`.

In [ ]:
RUN_EXPERIMENT = False  # Set True only when you are ready to launch training.
SMOKE = True            # First run True; use False for thesis validation results.
STAGE = "val"          # "val" or "test"
DEVICE = "cuda:0"      # "auto", "cpu", "cuda", or "cuda:N"
PROGRESS = "auto"      # "auto", "on", or "off"
VARIANTS = "A0,A1,A2,A4,A5"
SEED = 42
RESUME = False
CONFIG_FILE = "configs/mimic_ctabgan.json"
OUTPUT_DIR = None        # Optional explicit directory; otherwise fingerprinted.

## 2. Locate the project and verify the environment

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import json
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "xai_reweighting").is_dir() and (candidate / "configs").is_dir():
            return candidate
        nested = candidate / "CTAB-GAN-Plus-main"
        if (nested / "xai_reweighting").is_dir():
            return nested
    raise FileNotFoundError("Could not locate CTAB-GAN-Plus-main from the notebook directory")

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = (PROJECT_ROOT / CONFIG_FILE).resolve()
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_PATH = (PROJECT_ROOT / config["data_path"]).resolve()

print("Project:     ", PROJECT_ROOT)
print("Python:      ", sys.executable)
print("Configuration:", CONFIG_PATH)
print("Data:        ", DATA_PATH)

In [ ]:
import torch

gpu_info = {
    "PyTorch": torch.__version__,
    "CUDA build": torch.version.cuda,
    "CUDA available": torch.cuda.is_available(),
    "GPU count": torch.cuda.device_count(),
    "Selected GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
display(pd.Series(gpu_info, name="value").to_frame())

if RUN_EXPERIMENT and not DATA_PATH.is_file():
    raise FileNotFoundError(f"Required dataset not found: {DATA_PATH}")
if RUN_EXPERIMENT and DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable in the selected Jupyter kernel")
if RUN_EXPERIMENT and STAGE == "test" and not config.get("frozen", False):
    raise RuntimeError("Test runs require a reviewed configuration with frozen=true")
if RUN_EXPERIMENT and STAGE == "test" and SMOKE:
    raise RuntimeError("Smoke mode cannot be used for the frozen test stage")

## 3. Build and execute the CLI command

The cell streams combined stdout/stderr so epoch progress remains visible. An interrupted run can be continued by setting `RESUME=True`, provided its configuration, data, and code are unchanged.

In [ ]:
command = [
    sys.executable, "-u", "-m", "xai_reweighting.run_ablation",
    "--config", str(CONFIG_PATH),
    "--stage", STAGE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--variants", VARIANTS,
    "--progress", PROGRESS,
]
if SMOKE:
    command.append("--smoke")
if RESUME:
    command.append("--resume")
if OUTPUT_DIR:
    command.extend(["--output-dir", str(Path(OUTPUT_DIR).expanduser().resolve())])

print("Command:")
print(" ".join(command))

In [ ]:
run_directory = None
captured_lines = []

if RUN_EXPERIMENT:
    started = time.monotonic()
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        captured_lines.append(line.rstrip())
        print(line, end="", flush=True)
    return_code = process.wait()
    elapsed = time.monotonic() - started
    if return_code != 0:
        raise RuntimeError(f"Experiment failed with exit code {return_code}")
    for line in reversed(captured_lines):
        try:
            candidate = Path(line.strip())
            if candidate.is_absolute() and candidate.is_dir():
                run_directory = candidate.resolve()
                break
        except OSError:
            continue
    print(f"\nCLI completed in {elapsed / 60:.1f} minutes")
else:
    print("Training was not launched. Set RUN_EXPERIMENT=True and rerun this cell when ready.")

## 4. Select a completed result

After a run, its printed output directory is used automatically. Otherwise, this selects the newest matching result so the presentation section can be reopened without retraining.

In [ ]:
if run_directory is None:
    if OUTPUT_DIR and Path(OUTPUT_DIR).expanduser().is_dir():
        run_directory = Path(OUTPUT_DIR).expanduser().resolve()
    else:
        matches = sorted(
            (PROJECT_ROOT / config.get("results_dir", "results")).glob(
                f"mimic_ctabgan_{STAGE}_seed{SEED}_*"
            ),
            key=lambda path: path.stat().st_mtime,
        )
        if not matches:
            raise FileNotFoundError("No matching result directory exists yet")
        run_directory = matches[-1].resolve()

manifest = json.loads((run_directory / "manifest.json").read_text(encoding="utf-8"))
manifest_view = {
    key: manifest.get(key)
    for key in [
        "status", "device", "cuda_device_name", "precision", "deterministic",
        "smoke", "progress", "started_at", "completed_at",
        "elapsed_seconds", "variants_completed", "fingerprint",
    ]
}
display(Markdown(f"### Selected run: `{run_directory.name}`"))
display(pd.Series(manifest_view, name="value").to_frame())
if manifest.get("status") != "complete":
    display(Markdown("**Warning:** this run is not marked complete; some artifacts may be absent."))

## 5. Main ablation results

In [ ]:
summary = pd.read_csv(run_directory / "ablation_summary.csv")
key_metrics = [
    "variant", "utility_roc_auc", "utility_pr_auc", "utility_f1",
    "utility_recall", "rare_event_recall",
    "rare_outcome_frequency_error", "detector_auc",
    "detector_average_precision", "correlation_distance",
    "mean_wasserstein_scaled", "mean_cdf_tail_divergence",
    "privacy_exact_match_rate", "privacy_median_distance_ratio",
    "training_rows", "synthetic_rows", "weight_mean",
]
display(summary[[column for column in key_metrics if column in summary.columns]].round(4))

deltas_path = run_directory / "ablation_deltas.csv"
if deltas_path.exists():
    deltas = pd.read_csv(deltas_path)
    delta_focus = [
        "comparison", "delta_utility_roc_auc", "delta_utility_pr_auc",
        "delta_rare_event_recall", "delta_rare_outcome_frequency_error",
        "delta_detector_auc", "delta_mean_wasserstein_scaled",
        "delta_mean_cdf_tail_divergence",
    ]
    display(Markdown("### Incremental controlled comparisons"))
    display(deltas[[column for column in delta_focus if column in deltas.columns]].round(4))

numeric_summary = summary.set_index("variant").select_dtypes(include=np.number)
if {"A1", "A5"}.issubset(numeric_summary.index):
    display(Markdown("### Full method versus uniform augmentation: A5 − A1"))
    a5_a1 = (numeric_summary.loc["A5"] - numeric_summary.loc["A1"]).rename("delta")
    focus = [name for name in [
        "utility_roc_auc", "utility_pr_auc", "rare_event_recall",
        "rare_outcome_frequency_error", "detector_auc",
        "mean_wasserstein_scaled", "mean_cdf_tail_divergence",
        "privacy_exact_match_rate", "privacy_median_distance_ratio",
    ] if name in a5_a1.index]
    display(a5_a1.loc[focus].to_frame().round(4))

In [ ]:
plot_metrics = [
    metric for metric in [
        "utility_roc_auc", "utility_pr_auc", "rare_event_recall",
        "detector_auc", "rare_outcome_frequency_error",
        "mean_wasserstein_scaled", "mean_cdf_tail_divergence",
    ] if metric in summary.columns
]
if plot_metrics:
    columns = 2
    rows = int(np.ceil(len(plot_metrics) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, 4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, metric in zip(axes, plot_metrics):
        sns.barplot(data=summary, x="variant", y=metric, ax=axis, color="#4C78A8")
        axis.set_title(metric.replace("_", " ").title())
        axis.grid(axis="y", alpha=0.25)
    for axis in axes[len(plot_metrics):]:
        axis.remove()
    fig.suptitle("Ablation results", fontsize=16, y=1.01)
    fig.tight_layout()
    plt.show()

## 6. Feature priorities and weighting behavior

In [ ]:
score_frames = []
for variant in ("A2", "A4", "A5"):
    path = run_directory / f"feature_scores_{variant}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame.insert(0, "variant", variant)
        score_frames.append(frame)

if score_frames:
    scores = pd.concat(score_frames, ignore_index=True)
    selected_scores = scores[scores["selected"].astype(bool)].copy()
    display(selected_scores[[
        column for column in [
            "variant", "feature", "shap", "mismatch", "tail",
            "combined_raw", "priority",
        ] if column in selected_scores.columns
    ]].sort_values(["variant", "priority"], ascending=[True, False]).round(4))

    fig, axes = plt.subplots(1, len(score_frames), figsize=(6 * len(score_frames), 5), squeeze=False)
    for axis, (variant, frame) in zip(axes.ravel(), selected_scores.groupby("variant")):
        ordered = frame.sort_values("priority", ascending=True)
        axis.barh(ordered["feature"], ordered["priority"], color="#F58518")
        axis.set_title(f"{variant} selected feature priorities")
        axis.set_xlabel("Normalized priority")
    fig.tight_layout()
    plt.show()

In [ ]:
weight_frames = []
for variant in ("A2", "A4", "A5"):
    path = run_directory / f"row_weights_{variant}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["variant"] = variant
        weight_frames.append(frame[["variant", "weight"]])

if weight_frames:
    weights = pd.concat(weight_frames, ignore_index=True)
    display(weights.groupby("variant")["weight"].describe().round(4))
    plt.figure(figsize=(10, 5))
    sns.histplot(data=weights, x="weight", hue="variant", element="step", stat="density", common_norm=False)
    plt.title("Training-row weight distributions")
    plt.grid(axis="y", alpha=0.25)
    plt.show()

## 7. Per-feature diagnostics and artifact inventory

In [ ]:
FEATURE_VARIANT = "A5"
feature_path = run_directory / f"feature_metrics_{FEATURE_VARIANT}.csv"
if feature_path.exists():
    feature_metrics = pd.read_csv(feature_path)
    display(feature_metrics.round(4))

artifact_rows = []
for path in sorted(run_directory.iterdir()):
    if path.is_file() and not path.name.startswith("."):
        artifact_rows.append({
            "artifact": path.name,
            "size_mb": path.stat().st_size / (1024 ** 2),
        })
display(Markdown("### Saved artifacts"))
display(pd.DataFrame(artifact_rows).round({"size_mb": 3}))

## Interpretation reminders

- Higher utility AUC, PR-AUC, F1, and rare-event recall are better.
- Lower distribution, tail, rare-outcome frequency, and correlation errors are better.
- A detector AUC near 0.5 indicates that real and synthetic rows are difficult to distinguish, but it must be interpreted alongside utility and privacy.
- A lower exact-match rate and synthetic nearest-neighbor distances comparable to the held-out-real baseline are preferable privacy-proxy behavior.
- Use validation results for configuration decisions. Run the frozen test stage only once after those decisions are final.